# grc-joint — seed replicates of the shipped recipe

Trains the **exact stage-dplus-12 recipe** (GreBerta, 12 epochs, lr 3e-5, batch 32, max-len 256, bf16, combined AGDT+Gorman+Pedalion corpus) under **several different seeds**, evaluates each on the UD-Perseus and UD-PROIEL **test** folds, and reports **mean ± std**.

**Why:** the shipped model is seed 42 (Perseus LAS 84.38). The LAS lead over best-published (83.98) is only +0.40, within run-to-run noise. This notebook measures the real seed-only variance band so we can either back the SOTA-LAS claim honestly (if the band clears the bar) or state it as a tie.

**Before running:** `Runtime → Change runtime type → GPU`. The **G4 (NVIDIA RTX PRO 6000 Blackwell, 96 GB)** is the card this model was trained on — ideal.

**Runtime:** ≈12–15 min per seed on the G4 (the shipped 12-epoch run took ~14 min on this card and peaked at only 6.65 / 96 GB — training is compute-bound, not memory-bound, so the recipe is **kept identical**, never scaled to fill VRAM: that's what keeps the seeds comparable to the shipped model). 5 seeds ≈ 1–1.5 h. Reduce `SEEDS` if time-constrained (3 is the minimum for a usable std).

**When it finishes:** it downloads `seed_results.zip` — send that back (or just `seed-summary.json`) and I'll fold the mean ± std into `docs/benchmarks.md` and settle the LAS claim.

In [ ]:
# 1 — GPU check (expects a Colab G4 = NVIDIA RTX PRO 6000 Blackwell, 96 GB — the card the model was trained on)
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> GPU."
cap = torch.cuda.get_device_capability()
print(f"torch {torch.__version__} | {torch.cuda.get_device_name(0)} | sm_{cap[0]}{cap[1]}")
print(f"bf16 supported: {torch.cuda.is_bf16_supported()}  -> train_full.py auto-selects bf16 + TF32 on this card")
# Blackwell is sm_10x/sm_12x; if bf16 is unsupported here the torch build is too old for this GPU.
assert torch.cuda.is_bf16_supported(), "bf16 unsupported (unexpected on Blackwell) — the torch build is likely too old for sm_12x."

In [ ]:
# 2 — clone the repo (training scripts) and install
import importlib
%cd /content
!rm -rf pyaegean
!git clone --depth 1 https://github.com/ryanpavlicek/pyaegean.git
%cd /content/pyaegean
!pip install -q .             # NON-editable: copies aegean into site-packages (importable in the kernel AND in subprocesses)
!pip install -q transformers  # torch + numpy come preinstalled on Colab GPU runtimes
importlib.invalidate_caches()
import aegean, transformers, numpy
print("aegean", aegean.__version__, "| transformers", transformers.__version__, "| numpy", numpy.__version__)

In [ ]:
# 3 — build the combined dataset ONCE (seed-independent; fetches AGDT + UD folds + Gorman + Pedalion to cache)
%cd /content/pyaegean
!python training/build_full_dataset.py --with-extras
import os
assert os.path.exists("training/data/full-train.jsonl"), "dataset build did not produce full-train.jsonl"
print("dataset ready")

In [ ]:
# 4 — config
SEEDS = [1, 2, 3, 4, 5]      # reduce to [1, 2, 3] if GPU time is tight; the shipped model is seed 42 (Perseus LAS 84.38)
EPOCHS = 12
COMPOSE = "lookup-first"     # lemma composition only; does NOT affect LAS/UAS
import os
os.makedirs("/content/seed_results", exist_ok=True)

In [ ]:
# 5 — train + evaluate each seed (fresh subprocess per seed so VRAM is released between runs)
import subprocess, json, time
from pathlib import Path

def run(cmd):
    print(">>", " ".join(cmd)); t = time.time()
    r = subprocess.run(cmd, cwd="/content/pyaegean")
    if r.returncode != 0:
        raise RuntimeError(f"command failed ({r.returncode}): {' '.join(cmd)}")
    print(f"   done in {time.time() - t:.0f}s")

for s in SEEDS:
    out = f"/content/pyaegean/training/out/full-seed{s}"
    res = Path(f"/content/seed_results/seed{s}"); res.mkdir(parents=True, exist_ok=True)
    run(["python", "training/train_full.py", "--model", "bowphs/GreBerta",
         "--epochs", str(EPOCHS), "--lr", "3e-5", "--batch", "32", "--max-len", "256",
         "--seed", str(s), "--out", out])
    assert Path(f"{out}/model").exists(), f"no checkpoint at {out}/model"
    for tb in ("perseus", "proiel"):
        run(["python", "training/eval_full_ud.py", "--checkpoint", f"{out}/model",
             "--compose", COMPOSE, "--treebank", tb, "--split", "test",
             "--out", str(res / f"ud-{tb}-test.json")])
    p = json.loads((res / "ud-perseus-test.json").read_text())
    print(f"seed {s}: Perseus LAS {p['las']*100:.2f}  UAS {p['uas']*100:.2f}  lemma {p['lemma']*100:.2f}")

In [ ]:
# 6 — aggregate: mean +/- std per metric, and check the LAS claim
import json, statistics as st
from pathlib import Path

BEST_PUB = {"perseus": {"las": 0.8398, "uas": 0.8820}}   # Riemenschneider & Frank 2024
SHIPPED  = {"perseus": {"las": 0.8438, "uas": 0.8916}}   # seed 42, stage-e
folds = ("perseus", "proiel")
metrics = ("upos", "xpos", "ufeats", "lemma", "uas", "las")
summary = {}
for tb in folds:
    print(f"\n=== {tb} test ===")
    summary[tb] = {}
    for m in metrics:
        xs = []
        for s in SEEDS:
            p = Path(f"/content/seed_results/seed{s}/ud-{tb}-test.json")
            if p.exists():
                v = json.loads(p.read_text()).get(m)
                if isinstance(v, (int, float)) and v > 0:
                    xs.append(v)
        if not xs:
            continue
        mean = st.mean(xs); sd = st.pstdev(xs) if len(xs) > 1 else 0.0
        summary[tb][m] = {"mean": mean, "std": sd, "n": len(xs), "runs": xs}
        print(f"  {m:7s} {mean*100:6.2f} +/- {sd*100:.2f}   runs={[round(x*100, 2) for x in xs]}")

la = summary["perseus"].get("las")
if la:
    margin = (la["mean"] - BEST_PUB["perseus"]["las"]) * 100
    print(f"\nPerseus LAS: mean {la['mean']*100:.2f} +/- {la['std']*100:.2f} over {la['n']} seeds")
    print(f"  best published 83.98   ->  mean margin {margin:+.2f}")
    print(f"  shipped seed-42        ->  84.38")
    clears_2sd = la["mean"] - 2 * la["std"] > BEST_PUB["perseus"]["las"]
    print(f"  lead clears 2-sigma over best-published? {clears_2sd}")
    print("  -> True  => SOTA-LAS claim is defensible; False => report as a tie within noise")

Path("/content/seed_results/seed-summary.json").write_text(json.dumps(summary, indent=2))
print("\nwrote /content/seed_results/seed-summary.json")

In [ ]:
# 7 — download everything to send back
import shutil
shutil.make_archive("/content/seed_results", "zip", "/content/seed_results")
try:
    from google.colab import files
    files.download("/content/seed_results.zip")
except Exception as e:
    print("auto-download failed; grab /content/seed_results.zip from the file browser. (", e, ")")